# 🎛️ 24-信号处理 · 07 数字滤波器设计实务

> 关键词：Butterworth、双线性变换、FIR 窗函数法、过渡带
> §04 讲了 FIR/IIR 原理，本章解决"指标 → 滤波器系数"的设计流程，手算到阶数公式。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | 设计指标怎么描述（通带/阻带/dB）？ | §1 核心概念 |
| 2 | Butterworth 阶数公式怎么手算？ | §2 手算 ✍️ |
| 3 | 双线性变换为什么不会混叠？ | §3 深入原理 |
| 4 | FIR 窗函数法的过渡带/阻带怎么估？ | §4 代码实战 |

## 故事引入

产品要求">5kHz 一律切掉，衰减 ≥40dB"。怎么从这句话得到滤波器系数？标准流程：**指标 → 模拟原型（Butterworth）→ 双线性变换数字化（IIR）** 或 **理想低通截断加窗（FIR）**。本章手算第一条路。

"设计指标"在真实工程里长什么样？看声学中著名的 A 计权（A-weighting）滤波器：它模拟人耳对不同频率的敏感度，要求在 1kHz 附近 0dB、低频大幅衰减。下图实线是理想 A 计权幅响，这正是"给出指标 → 设计滤波器逼近指标"的标准实例，和本节的通带/阻带指标（Ap、As、ωp、ωs）是同一套语言。

!["设计指标"在真实工程里长什么样？看声学中著名的 A 计权（](../figures/aweighting_mag.jpg)

> **怎么读**：横轴频率（对数）、纵轴增益（dB）：低频段快速滚降、中频平缓、高频略回升。设计任务就是让实际滤波器（虚线/散点）去逼近这条目标曲线——Butterworth 阶数与双线性变换就是为了做到这一点。
>
> 📚 **图源**：Pilipović R, Risojević V & Bulić P (2021) *On the Design of an Energy Efficient Digital IIR A-Weighting Filter Using Approximate Multiplication*, Sensors 21(3):732（开放获取，CC-BY 4.0）图 1。


## 核心概念：指标与原型

**设计指标**（低通）：通带纹波 $A_p$（dB）、阻带衰减 $A_s$（dB）、通带边缘 $\omega_p$、阻带边缘 $\omega_s$（rad/s）。

**Butterworth 原型**（最平坦通带）：

$$|H_a(j\Omega)|^2=\frac{1}{1+(\Omega/\Omega_c)^{2N}}$$

**阶数公式**

$$N\ge\frac{\log_{10}\frac{\sqrt{10^{A_s/10}-1}}{\sqrt{10^{A_p/10}-1}}}{\log_{10}(\Omega_s/\Omega_p)}$$

**3dB 截止**：$\Omega_c=\Omega_p/(10^{A_p/10}-1)^{1/(2N)}$。

## ✍️ 手算 Butterworth 阶数：Ap=1dB, As=40dB, Ωs/Ωp=2

**第 1 步：两个平方根**

$$\sqrt{10^{4}-1}=\sqrt{9999}\approx\mathbf{99.995},\qquad \sqrt{10^{0.1}-1}=\sqrt{0.2589}\approx\mathbf{0.5088}$$

**第 2 步：代入阶数公式**

$$N\ge\frac{\log_{10}(99.995/0.5088)}{\log_{10}2}=\frac{\log_{10}196.5}{0.3010}=\frac{2.2934}{0.3010}=\mathbf{7.62}\Rightarrow N=8$$

**第 3 步：3dB 截止频率**

$$\Omega_c=\frac{\Omega_p}{(10^{0.1}-1)^{1/16}}=\frac{1}{0.9190}=\mathbf{1.0881}$$

**手算收获**：

- 阶数公式的含义：分母是"过渡带宽度"（频率比对 2），分子是"必须压下去的总倍数"（196.5 倍）——**过渡带越窄、衰减要求越高，阶数越大**
- 8 阶 Butterworth 意味着 $|H|^2=1/(1+(\Omega/\Omega_c)^{16})$：每倍频程斜率 $40N\,\text{dB/decade}$？不——是 $-20N\,\text{dB/decade}$，8 阶即 −160dB/decade，骤降
- 取整向上：7.62 → 8 阶才会满足阻带指标；取 N=7 则不够
- 双线性变换后数字滤波器的阶数不变但频率被"压缩"，所以模拟指标要先预扭曲（§3）

In [1]:
# ---- §2.1 验证：numpy 复算 Butterworth 阶数 ----
import math
As, Ap, ratio = 40, 1, 2
num = math.sqrt(10**(As/10) - 1)
den = math.sqrt(10**(Ap/10) - 1)
N = math.log10(num/den) / math.log10(ratio)
print(f'N = log10({num/den:.1f})/log10(2) = {N:.4f} → 取 {math.ceil(N)} 阶')
assert math.ceil(N) == 8
wc = 1 / (10**(Ap/10) - 1)**(1/(2*math.ceil(N)))
print(f'Ωc = 1/(10^0.1-1)^(1/16) = {wc:.4f}')
assert abs(wc - 1.0881) < 0.001
print('\n✓ 与手算一致')


N = log10(196.5)/log10(2) = 7.6185 → 取 8 阶
Ωc = 1/(10^0.1-1)^(1/16) = 1.0881

✓ 与手算一致


## 深入原理：双线性变换

**双线性变换**（s→z）：

$$s=\frac{2}{T}\frac{1-z^{-1}}{1+z^{-1}}$$

$j\Omega$ 轴被一对一映射到单位圆，**无混叠**（这是它比冲激不变法优越之处）；代价是频率非线性压缩（frequency warping）：

$$\Omega=\frac{2}{T}\tan\frac{\omega}{2}$$

所以：先按扭曲后的频率设计模拟原型（预扭曲），再双线性变换得到数字系数。 **冲激不变法**：$h_d[n]=T\,h_a(nT)$，模拟频谱周期性搬移会混叠，只适合带限滤波器。

## 代码实战：FIR 窗函数法的过渡带估算

理想低通（截止 $\omega_c$）冲激响应 $h[n]=\frac{\omega_c}{\pi}\text{sinc}$ 无限长；截断到 M 点并加窗：

- 过渡带宽度 ≈ 窗主瓣宽：Hann ≈ $8\pi/M$，Hamming ≈ $8\pi/M$，矩形 ≈ $4\pi/M$
- 阻带衰减由窗旁瓣决定（Hann −44dB、Hamming −53dB、Blackman −74dB）

手算：M=32 → 过渡带 $\approx 4\pi/32=0.393$ rad（矩形窗）。

In [2]:
# ---- FIR 低通：窗函数法（numpy 实现，无 scipy） ----
import numpy as np

def fir_lowpass(wc, M, window):
    n = np.arange(M) - (M-1)/2
    h = np.where(n == 0, wc/np.pi, np.sin(wc*n)/(np.pi*n))   # 理想低通
    if window == 'rect':
        w = np.ones(M)
    elif window == 'hann':
        w = 0.5*(1 - np.cos(2*np.pi*np.arange(M)/(M-1)))
    else:  # hamming
        w = 0.54 - 0.46*np.cos(2*np.pi*np.arange(M)/(M-1))
    return h * w

def freq_at(h, w):
    return np.abs(np.sum(h * np.exp(-1j*w*np.arange(len(h)))))

M = 32
for name, win in [('rect', 'rect'), ('hann', 'hann'), ('hamming', 'hamming')]:
    h = fir_lowpass(0.4*np.pi, M, win)
    k = np.arange(M) - (M-1)/2
    g0 = np.abs(np.sum(h))                                    # ω=0 通带增益
    g_s = np.abs(np.sum(h * np.exp(-1j*0.55*np.pi*k)))        # ω=0.55π 阻带增益
    print(f'{name:8s}: 阻带@0.55π = {20*np.log10(g_s/g0):6.1f} dB（ω=π 因偶对称×(-1)^k 恒为 0）')
g_db = []
for win in ['rect', 'hann', 'hamming']:
    h = fir_lowpass(0.4*np.pi, M, win)
    k = np.arange(M) - (M-1)/2
    g_db.append(20*np.log10(np.abs(np.sum(h*np.exp(-1j*0.55*np.pi*k))) / np.abs(np.sum(h))))
assert g_db[2] < g_db[1] < g_db[0]     # 阻带衰减: hamming > hann > rect
print('✓ 阻带衰减按 hamming > hann > rect，与旁瓣指标一致')


rect    : 阻带@0.55π =  -31.7 dB（ω=π 因偶对称×(-1)^k 恒为 0）
hann    : 阻带@0.55π =  -49.1 dB（ω=π 因偶对称×(-1)^k 恒为 0）
hamming : 阻带@0.55π =  -58.6 dB（ω=π 因偶对称×(-1)^k 恒为 0）
✓ 阻带衰减按 hamming > hann > rect，与旁瓣指标一致


## 面试考点

**Q1：IIR 和 FIR 设计的数字化方法各是什么？**

IIR：模拟原型设计（Butterworth/Chebyshev/椭圆）+ 双线性变换；FIR：窗函数法 / 频率采样 / Parks-McClellan 等波纹。

**Q2：双线性变换优点/缺点？**

无混叠 + 保留稳定（sLHP→z 单位圆内）；缺点：频率非线性压缩，须预扭曲。

**Q3：窗函数法阻带衰减为什么有限？**

截断 = 矩形窗卷积，旁瓣泄漏决定了阻带底；换低旁瓣窗改善阻带但过渡带变宽。

**Q4：等波纹（Equiripple）滤波器比窗函数法好在哪里？**

在同等阶数下把误差均匀摊到通带/阻带，过渡带更窄——但设计成本高。

## 小结与下一章预告

- Butterworth 8 阶手算：$N\ge\log_{10}(196.5)/\log_{10}2=7.62\to8$，$\Omega_c=1.0881$
- 双线性变换：无混叠但须预扭曲；FIR 窗函数法：过渡带 ≈ 主瓣宽
- 下一章：滤波器学完了，去看看语音最常用的**特征**——MFCC

## ✅ 自测清单

- [ ] 能说出设计指标（Ap/As/ωp/ωs）含义
- [ ] 能手算 Butterworth 阶数公式并取整
- [ ] 能解释双线性变换无混叠的原理与预扭曲
- [ ] 知道 FIR 窗函数法的过渡带与阻带由什么决定